## Setup and Dependencies

### Library

In [1]:
!pip install -q timesfm[torch]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00


In [54]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import auth
from google.colab import drive

import timesfm

from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

### Auth

In [3]:
drive.mount('/content/drive')

auth.authenticate_user()

### Loader

Query for access data in Google BigQuery

## Data Preparation

Load data from bigquery

In [6]:
df_50c = pd.read_gbq(sql_query_50c, project_id=project_id, dialect='standard')
df_150c = pd.read_gbq(sql_query_150c, project_id=project_id, dialect='standard')

/tmp/ipykernel_1033/3761027708.py:1: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df_50c = pd.read_gbq(sql_query_50c, project_id=project_id, dialect='standard')
/tmp/ipykernel_1033/3761027708.py:2: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df_150c = pd.read_gbq(sql_query_150c, project_id=project_id, dialect='standard')


### Exploration

Checking unique value

In [15]:
unique_values_50c = df_50c.nunique()
unique_values_150c = df_150c.nunique()

print("")
print("====== Data 50 Cohorts =====")
print(unique_values_50c)

print("")
print("====== Data 150 Cohorts ======")
print(unique_values_150c)


====== Data 50 Cohorts =====
date_created_at            42
adnet                       5
date_update               519
no_of_user                435
revenue_dailypush         368
subject                     3
revenue_running_total    5247
dtype: int64

====== Data 150 Cohorts ======
date_created_at           186
adnet                       5
date_update               532
no_of_user                551
revenue_dailypush         399
subject                     3
revenue_running_total    6672
dtype: int64


Checking duplicate data

In [19]:
print("Duplicated data in 50 cohorts",df_50c.duplicated().sum())
print("Duplicated data in 150 cohorts",df_150c.duplicated().sum())

Duplicated data in 50 cohorts 0
Duplicated data in 150 cohorts 0


### Feature Engineering

Adding cohort age (date update - date created) for getting information day every cohort

In [22]:
df_50c['cohort_age'] = (
    df_50c['date_update'] -
    df_50c['date_created_at']
).dt.days

In [23]:
df_150c['cohort_age'] = (
    df_150c['date_update'] -
    df_150c['date_created_at']
).dt.days

## Preprocessing

### Sort Data

Final preparation with sorting date created and cohort age

In [24]:
def prepare_data(df):
    return (
        df
        .sort_values(['date_created_at', 'cohort_age'])
        .reset_index(drop=True)
    )

In [25]:
df_50c = prepare_data(df_50c)
df_150c = prepare_data(df_150c)

In [26]:
df_150c.head()

,date_created_at,adnet,date_update,no_of_user,revenue_dailypush,subject,revenue_running_total,cohort_age
0,2025-01-01,ADN,2025-01-01,2662,1054000.0,FIRSTPUSH,1054000.0,0
1,2025-01-01,C2M,2025-01-01,86,6000.0,FIRSTPUSH,6000.0,0
2,2025-01-01,MOBIP,2025-01-01,596,68000.0,FIRSTPUSH,68000.0,0
3,2025-01-01,MBP,2025-01-01,686,136000.0,FIRSTPUSH,136000.0,0
4,2025-01-01,C2M,2025-01-02,0,0.0,NO_ACTIVITY,6000.0,1


### Feature Selection

Selecting feature for new dataframe

In [27]:
KEEP_COLS = [
    'date_created_at',
    'adnet',
    'cohort_age',
    'revenue_running_total',
]

df_model_50c = df_50c[KEEP_COLS].copy()
df_model_150c = df_150c[KEEP_COLS].copy()

Setup feature and target for training

In [33]:
TARGET = 'revenue_running_total'

## Forecast Setup

### Series Builder

In [ ]:
CONTEXT_LEN = 2048   # max context length (2.5 supports up to 16k)
HORIZON_LEN = 512    # max forecast horizon per call
CONTEXT_MIN = 0     # minimum history required before forecasting
# Note: TimesFM 2.5 removed the frequency indicator — no FREQ needed

def get_series(data, cohort, adnet):
    """
    Return the ordered univariate target series for one
    (date_created_at, adnet).
    """

    s = (
        data[
            (data["date_created_at"] == cohort) &
            (data["adnet"] == adnet)
        ]
        .sort_values("cohort_age")
    )

    return s[TARGET].to_numpy(dtype=float)

## Modeling

### TimesFM

TimesFM is a pretrained foundation model, so it is used zero-shot — there is no training loop. Load the 2.5 200M PyTorch checkpoint with the fixed architecture parameters.

In [30]:
import torch
import timesfm

torch.set_float32_matmul_precision("high")

# TimesFM 2.5 – 200M parameter checkpoint (PyTorch)
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

tfm.compile(
    timesfm.ForecastConfig(
        max_context=CONTEXT_LEN,
        max_horizon=HORIZON_LEN,
        normalize_inputs=True,
        use_continuous_quantile_head=False,   # point forecast only
        force_flip_invariance=False,
        infer_is_positive=True,               # revenue is always >= 0
        fix_quantile_crossing=True,
    )
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/475 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]

## Evaluation

### Visualization

Evaluate zero-shot on the validation cohorts: for each cohort use all but the last `EVAL_HORIZON` days as context and forecast the held-out tail, then compare against the actuals.

In [ ]:
def evaluate_forecast(df_model, seed_days, max_cohort_age=180, output_csv=None):
    """
    Evaluate TimesFM forecast on validation cohorts.
    
    Args:
        df_model: DataFrame
        seed_days: Number of days as input
        max_cohort_age: Maximum cohort age
        output_csv: Path to save CSV
    
    Returns:
        Dictionary with metrics
    """
    eval_horizon = max_cohort_age - seed_days
    milestone_days = [60, 90, 120, 150, 180]
    
    eval_contexts = []
    eval_actuals = []
    eval_details = []
    
    all_cohorts = sorted(df_model["date_created_at"].unique())
    
    print("="*80)
    print(f"Evaluation Setup:")
    print(f"  Input (seed_days): {seed_days} days")
    print(f"  Max cohort age: {max_cohort_age} days")
    print(f"  Eval horizon: {eval_horizon} days")
    print(f"  Total cohorts: {len(all_cohorts)}")
    
    for cohort in all_cohorts:
        adnets = (
            df_model.loc[
                df_model["date_created_at"] == cohort,
                "adnet"
            ]
            .unique()
        )
        
        for adnet in adnets:
            s = get_series(df_model, cohort, adnet)
            
            if len(s) < seed_days:
                continue
            
            actual_length = len(s)
            forecast_length = min(actual_length - seed_days, eval_horizon)
            
            if forecast_length <= 0:
                continue
            
            context = s[:seed_days]
            actual = s[seed_days:seed_days + forecast_length]
            
            eval_contexts.append(context)
            eval_actuals.append(actual)
            
            last_day_idx = seed_days + forecast_length - 1
            actual_last_val = s[last_day_idx]
            
            eval_details.append({
                "cohort": cohort,
                "adnet": adnet,
                "last_day": last_day_idx,
                "actual_last_val": actual_last_val,
                "full_series": s,  # Keep full series untuk milestone
            })
    
    print(f"Evaluated series: {len(eval_contexts)}\n")
    
    if len(eval_contexts) == 0:
        print("⚠️ No series to evaluate!")
        return None
    
    max_forecast_len = max(len(a) for a in eval_actuals)
    
    point_forecast, _ = tfm.forecast(
        horizon=max_forecast_len,
        inputs=eval_contexts,
    )
    
    # Collect predictions
    y_pred_list = []
    pred_last_values = []
    
    for i, actual_len in enumerate([len(a) for a in eval_actuals]):
        pred = point_forecast[i, :actual_len]
        y_pred_list.append(pred)
        pred_last_values.append(pred[-1])
    
    y_pred = np.concatenate(y_pred_list)
    y_val = np.concatenate(eval_actuals)
    
    # Calculate global metrics
    mae = mean_absolute_error(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mape = mean_absolute_percentage_error(y_val, y_pred) * 100
    
    # Calculate per-cohort metrics with milestones
    results_data = []
    
    for i, detail in enumerate(eval_details):
        actual_val = eval_actuals[i]
        pred_val = y_pred_list[i]
        full_series = detail["full_series"]
        
        # MAE per cohort
        mae_cohort = mean_absolute_error(actual_val, pred_val)
        
        # MAPE per cohort
        if np.any(actual_val == 0):
            mape_cohort = np.mean(np.abs((actual_val - pred_val) / (np.abs(actual_val) + 1))) * 100
        else:
            mape_cohort = mean_absolute_percentage_error(actual_val, pred_val) * 100
        
        # Build row
        row = {
            "Cohort": str(detail["cohort"])[:10],
            "Adnet": detail["adnet"],
            "Last_Day": detail["last_day"],
            "Actual_Last": int(detail["actual_last_val"]),
            "Predicted_Last": int(pred_last_values[i]),
            "MAE": f"{mae_cohort:,.2f}",
            "MAPE": f"{mape_cohort:.2f}%",
        }
        
        # Add milestone columns
        for milestone in milestone_days:
            if milestone < len(full_series):
                actual_milestone = int(full_series[milestone])
                pred_milestone = int(pred_val[milestone - seed_days]) if (milestone - seed_days) < len(pred_val) else None
            else:
                actual_milestone = None
                pred_milestone = None
            
            row[f"Actual_Day{milestone}"] = actual_milestone
            row[f"Predicted_Day{milestone}"] = pred_milestone
        
        results_data.append(row)
    
    # Create DataFrame
    results_df = pd.DataFrame(results_data)
    
    # Save to CSV
    if output_csv is None:
        output_csv = f"evaluation_seed{seed_days}_maxage{max_cohort_age}.csv"
    
    results_df.to_csv(output_csv, index=False)
    
    print(f"Results saved to: {output_csv}")
    print(f"\nGLOBAL EVALUATION RESULTS")
    print(f"  Seed: {seed_days}d | Max age: {max_cohort_age}d | Eval horizon: {eval_horizon}d")
    print(f"MAE  : {mae:>15,.2f}")
    print(f"RMSE : {rmse:>15,.2f}")
    print(f"MAPE : {mape:>14.2f}%")
    
    return {
        "mae": mae,
        "rmse": rmse,
        "mape": mape,
        "y_pred": y_pred,
        "y_val": y_val,
        "results_df": results_df,
    }


In [66]:
df_model_50c = prepare_data(df_50c)
df_model_150c = prepare_data(df_150c)
OUTPUT_DIR = "/content/drive/MyDrive/ltv-results"

In [67]:
eval_50c_7 = evaluate_forecast(
    df_model_50c, 
    seed_days=SEED_DAYS_WEEK, 
    max_cohort_age=MAX_COHORT_AGE,
    output_csv=os.path.join(OUTPUT_DIR, "eval_50c_seed7.csv")
)

eval_50c_30 = evaluate_forecast(
    df_model_50c, 
    seed_days=SEED_DAYS_MONTH, 
    max_cohort_age=MAX_COHORT_AGE,
    output_csv=os.path.join(OUTPUT_DIR, "eval_50c_seed30.csv")
)

Evaluation Setup:
  Input (seed_days): 7 days
  Max cohort age: 180 days
  Eval horizon: 173 days
  Total cohorts: 42
Evaluated series: 129

Results saved to: /content/drive/MyDrive/ltv-results/eval_50c_seed7.csv

GLOBAL EVALUATION RESULTS
  Seed: 7d | Max age: 180d | Eval horizon: 173d
MAE  :    3,557,039.71
RMSE :    9,715,715.30
MAPE :          73.14%
Evaluation Setup:
  Input (seed_days): 30 days
  Max cohort age: 180 days
  Eval horizon: 150 days
  Total cohorts: 42
Evaluated series: 129

Results saved to: /content/drive/MyDrive/ltv-results/eval_50c_seed30.csv

GLOBAL EVALUATION RESULTS
  Seed: 30d | Max age: 180d | Eval horizon: 150d
MAE  :    2,713,798.08
RMSE :    8,942,068.70
MAPE :          37.21%


In [68]:
eval_150c_7 = evaluate_forecast(
    df_model_150c, 
    seed_days=SEED_DAYS_WEEK, 
    max_cohort_age=MAX_COHORT_AGE,
    output_csv=os.path.join(OUTPUT_DIR, "eval_150c_seed7.csv")
)

eval_150c_30 = evaluate_forecast(
    df_model_150c, 
    seed_days=SEED_DAYS_MONTH, 
    max_cohort_age=MAX_COHORT_AGE,
    output_csv=os.path.join(OUTPUT_DIR, "eval_150c_seed30.csv")
)

Evaluation Setup:
  Input (seed_days): 7 days
  Max cohort age: 180 days
  Eval horizon: 173 days
  Total cohorts: 186
Evaluated series: 247

Results saved to: /content/drive/MyDrive/ltv-results/eval_150c_seed7.csv

GLOBAL EVALUATION RESULTS
  Seed: 7d | Max age: 180d | Eval horizon: 173d
MAE  :    2,763,497.29
RMSE :    8,190,449.35
MAPE :          72.79%
Evaluation Setup:
  Input (seed_days): 30 days
  Max cohort age: 180 days
  Eval horizon: 150 days
  Total cohorts: 186
Evaluated series: 247

Results saved to: /content/drive/MyDrive/ltv-results/eval_150c_seed30.csv

GLOBAL EVALUATION RESULTS
  Seed: 30d | Max age: 180d | Eval horizon: 150d
MAE  :    1,963,534.17
RMSE :    7,379,800.79
MAPE :          36.66%
